<div class="alert alert-block alert-info">

## Imports & Connection
</div>

In [64]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from IPython.display import display
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [21]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [22]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [23]:
print('Notebook dir - ', os.getcwd())
local_extract = '/Users/E2074/local-datasets/customer/ads-monetisation/holdout-group/experiment-result'
print('Data dir - ', local_extract)

Notebook dir -  /Users/E2074/analytics/customer/Ads-Monetisation/holdout-group/jas
Data dir -  /Users/E2074/local-datasets/customer/ads-monetisation/holdout-group/experiment-result


<div class="alert alert-block alert-info">
    
## Datasets & Parameter
</div>

In [160]:
## Parameter 
start_date = '2024-08-05'
end_date = '2024-09-29'
experiment_name = 'Ads-Holdout-Group-Experiment'
group_tc = ['control', 'test']
retention_segment = ['HH', 'GOLD', 'PLATINUM', 'DORMANT', 'UNKNOWN', 'ELITE', 'SILVER', 'PRIME']

In [25]:
def get_mondays_between(start_date, end_date):
    start = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    
    # Adjust start to the next Monday if it's not a Monday
    if start.weekday() != 0:  # 0 is Monday
        start += timedelta(days=(7 - start.weekday()))

    # Generate all Mondays between start and end dates
    mondays = []
    while start <= end:
        mondays.append(start.strftime('%Y-%m-%d'))
        start += timedelta(weeks=1)

    return mondays

week_start_date = get_mondays_between(start_date, end_date)
print(week_start_date)

['2024-08-05', '2024-08-12', '2024-08-19', '2024-08-26', '2024-09-02', '2024-09-09', '2024-09-16', '2024-09-23']


<div class="alert alert-block alert-info">
    
### Data
</div>

In [28]:
def get_weekly_retention_data(experiment_name, group_tc, week_start_date):

    df = pd.DataFrame()
    
    for date in week_start_date:

        for sample_category in group_tc:
            
            base_query = f"""
    
            WITH base AS (
                    SELECT 
                        DISTINCT
                        DATE_FORMAT(DATE_TRUNC('WEEK',DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y-%m-%d') yyyymmdd_base_week,
                        sample_category,
                        customer_id,
                        execution_name 
                    FROM
                        iceberg.experiments_internal.rapidex_user_customer_adslots_v1_root
                    WHERE
                        yyyymmdd >= DATE_FORMAT(DATE_TRUNC('WEEK', DATE('{date}')), '%Y%m%d')
                        AND yyyymmdd <= DATE_FORMAT(DATE_TRUNC('WEEK', DATE('{date}')) + INTERVAL '6' DAY , '%Y%m%d')
                        AND experiment_name = '{experiment_name}'
                        AND sample_category = '{sample_category}'
                ),
                
                activity AS (
                    SELECT 
                        DISTINCT
                        DATE_FORMAT(DATE_TRUNC('week', DATE_PARSE(day, '%Y-%m-%d')),'%Y-%m-%d') as date_period,
                        yyyymmdd_base_week,
                        customerid customer_id,
                        execution_name,
                        sample_category
                    FROM 
                        datasets.customer_rf_daily_kpi
                    JOIN 
                        base
                        ON customer_id = customerid
                    WHERE 
                        day >= DATE_FORMAT(DATE_TRUNC('WEEK', DATE('{date}')), '%Y-%m-%d')
                        AND day <= DATE_FORMAT(DATE_TRUNC('WEEK', CURRENT_DATE) - INTERVAL '1' DAY, '%Y-%m-%d')
                        AND service_name IN ('Link', 'CabPremium', 'Auto Pool', 'Bike Lite', 'Auto', 'Bike Metro', 'CabEconomy', 'Auto NCR', 'Bike Pink')
                        AND net_rides_daily > 0
                    -- GROUP BY 1,2,3
                ),
                
                customer_totals AS (
                    SELECT 
                        execution_name,
                        sample_category,
                        date_period, 
                        COUNT(DISTINCT customer_id) AS total_cx
                    FROM 
                        activity
                    WHERE 
                        date_period = yyyymmdd_base_week
                    GROUP BY 1,2,3
                ),
                
                customer_counts AS (
                    SELECT 
                        a1.execution_name, 
                        a1.sample_category,
                        a1.date_period AS date_period1,
                        a2.date_period AS date_period2,
                        COUNT(DISTINCT a1.customer_id) AS count_t1,
                        COUNT(DISTINCT a2.customer_id) AS count_t2
                    FROM 
                        activity a1
                    LEFT JOIN 
                        activity a2 
                        ON a1.customer_id = a2.customer_id 
                        AND a1.execution_name = a2.execution_name
                        AND a1.sample_category = a2.sample_category
                        AND a1.date_period <= a2.date_period
                    GROUP BY 1,2,3,4
                )
                
                SELECT 
                    customer_counts.execution_name,
                    customer_counts.sample_category,
                    date_period1 AS start_date, 
                    date_period2 AS retention_date, 
                    CASE 
                        WHEN date_period1 = date_period2 THEN 100.00 
                        ELSE (100.00 * customer_counts.count_t1) / customer_totals.total_cx 
                    END AS retention_percentage
                FROM 
                    customer_counts 
                JOIN 
                    customer_totals 
                    ON customer_totals.date_period = customer_counts.date_period1
                    AND customer_totals.execution_name = customer_counts.execution_name
                    AND customer_totals.sample_category = customer_counts.sample_category
                WHERE 
                    date_period2 IS NOT NULL
                ORDER BY 1,2,4,3
    
            """
            
            df_base = pd.read_sql(base_query, connection)
            df = pd.concat([df, df_base], ignore_index=True)
            print(sample_category, ' data fetched successfully for ', date)

    return df

In [ ]:
df_overall_data = get_weekly_retention_data(experiment_name, group_tc, week_start_date)
df_overall_data.to_csv(local_extract + '/city_level_experiment_retention_data.csv', index=False)

In [88]:
# Mapping dictionary
execution_city_name_mapping = {
    'holdout_segment_ban': 'Bangalore',
    'holdout_segment_che': 'Chennai',
    'holdout_segment_del': 'Delhi',
    'holdout_segment_hyd': 'Hyderabad',
    'holdout_segment_jai': 'Jaipur',
    'holdout_segment_kol': 'Kolkata',
    'holdout_segment_mum': 'Mumbai'
}

In [90]:
df_overall_data = pd.read_csv(local_extract + '/city_level_experiment_retention_data.csv')
print(df_overall_data.shape)
df_overall_data['city_name'] = df_overall_data['execution_name'].map(execution_city_name_mapping)
df_overall_data.head(5)

(984, 5)


,execution_name,sample_category,start_date,retention_date,retention_percentage,city_name
0,holdout_segment_ban,control,2024-08-05,2024-08-05,100.00,Bangalore
1,holdout_segment_ban,control,2024-08-05,2024-08-12,51.73,Bangalore
2,holdout_segment_ban,control,2024-08-05,2024-08-19,48.67,Bangalore
3,holdout_segment_ban,control,2024-08-05,2024-08-26,46.56,Bangalore
4,holdout_segment_ban,control,2024-08-05,2024-09-02,45.70,Bangalore


<div class="alert alert-block alert-info">
    
## City level Analysis
</div>

In [141]:
def get_city_retention_cohort(df, city_name):
    
    df_control = df[(df['city_name'] == city_name) & (df['sample_category'] == 'control')]
    df_test = df[(df['city_name'] == city_name) & (df['sample_category'] == 'test')]
    df_merge = pd.merge(df_control,df_test, how='inner',on=['city_name', 'start_date', 'retention_date'])
    df_merge['retention_delta'] =  df_merge['retention_percentage_x'] - df_merge['retention_percentage_y']
    df_merge['difference'] = 'delta - (control - test)'
    df1 = df_control.pivot(index=['start_date'], 
                           columns=['city_name', 'sample_category', 'retention_date'],
                           values='retention_percentage').fillna('')
    df2 = df_test.pivot(index=['start_date'], 
                        columns=['city_name', 'sample_category', 'retention_date'],
                        values='retention_percentage').fillna('')
    df3 = df_merge.pivot(index=['start_date'], 
                         columns=['city_name', 'difference', 'retention_date'], 
                         values='retention_delta').fillna('')
    print('standard deviation → ', round(df_merge['retention_delta'].std(), 2))
    
    return display(df1,df2,df3)

### Bangalore

In [142]:
get_city_retention_cohort(df_overall_data, 'Bangalore')

standard deviation -  0.2


city_name        Bangalore                                              \
sample_category    control                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      51.73      48.67      46.56       45.7   
2024-08-12                      100.0      50.04      45.88      44.86   
2024-08-19                                 100.0      50.25      47.04   
2024-08-26                                            100.0      51.24   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           45.23      43.73      42.68      43.66      42.99   
2024-08-12           44.32      42.77      41.39      42.51      42.16   
2024-08-19           46.03      44.42      42.97      43.60      42.93   
2024-08-26           48.29      46.36      44.97      45.37      44.16   
2024-09-02           51.34      47.43      45.42      46.14      45.26   
2024-09-09           100.0      50.85      46.70      46.67      45.58   
2024-09-16                      100.0      50.89      48.78      46.97   
2024-09-23                                100.00      52.73      48.76   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           42.77      41.48      38.17  
2024-08-12           41.76      40.30      37.91  
2024-08-19           42.67      41.40      38.34  
2024-08-26           43.98      42.69      39.18  
2024-09-02           44.27      42.76      40.11  
2024-09-09           44.83      43.24      40.00  
2024-09-16           46.11      44.39      40.69  
2024-09-23           47.61      46.00      41.68

city_name        Bangalore                                              \
sample_category       test                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      51.86      48.19      46.01      45.26   
2024-08-12                      100.0      49.88      45.59      45.18   
2024-08-19                                 100.0      49.96      47.26   
2024-08-26                                            100.0      51.22   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           44.93      43.57      42.95      43.34      42.93   
2024-08-12            44.2      43.06      41.65      42.66      42.27   
2024-08-19            45.8      44.64      43.41      43.30      43.08   
2024-08-26           48.43      46.25      45.42      45.12      44.05   
2024-09-02           51.35      47.63      45.61      46.11      45.39   
2024-09-09           100.0      50.71      46.75      46.57      45.65   
2024-09-16                      100.0      50.95      48.66      46.60   
2024-09-23                                100.00      52.64      48.57   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           42.95      41.67      38.29  
2024-08-12           41.71      40.31      37.93  
2024-08-19           42.66      41.41      38.24  
2024-08-26           43.86      42.64      39.46  
2024-09-02           44.17      42.83      39.94  
2024-09-09           44.88      43.10      39.98  
2024-09-16           45.98      44.37      40.56  
2024-09-23           47.61      46.32      41.69

city_name                     Bangalore                                   \
difference     delta - (test - control)                                    
retention_date               2024-08-05 2024-08-12 2024-08-19 2024-08-26   
start_date                                                                 
2024-08-05                          0.0       0.13      -0.48      -0.55   
2024-08-12                                     0.0      -0.16      -0.29   
2024-08-19                                                0.0      -0.29   
2024-08-26                                                           0.0   
2024-09-02                                                                 
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

city_name                                                              \
difference                                                              
retention_date 2024-09-02 2024-09-09 2024-09-16 2024-09-23 2024-09-30   
start_date                                                              
2024-08-05          -0.44       -0.3      -0.16       0.27      -0.32   
2024-08-12           0.32      -0.12       0.29       0.26       0.15   
2024-08-19           0.22      -0.23       0.22       0.44      -0.30   
2024-08-26          -0.02       0.14      -0.11       0.45      -0.25   
2024-09-02            0.0       0.01        0.2       0.19      -0.03   
2024-09-09                       0.0      -0.14       0.05      -0.10   
2024-09-16                                  0.0       0.06      -0.12   
2024-09-23                                            0.00      -0.09   

city_name                                                   
difference                                                  
retention_date 2024-10-07 2024-10-14 2024-10-21 2024-10-28  
start_date                                                  
2024-08-05          -0.06       0.18       0.19       0.12  
2024-08-12           0.11      -0.05       0.01       0.02  
2024-08-19           0.15      -0.01       0.01      -0.10  
2024-08-26          -0.11      -0.12      -0.05       0.28  
2024-09-02           0.13      -0.10       0.07      -0.17  
2024-09-09           0.07       0.05      -0.14      -0.02  
2024-09-16          -0.37      -0.13      -0.02      -0.13  
2024-09-23          -0.19       0.00       0.32       0.01

### Chennai

In [143]:
get_city_retention_cohort(df_overall_data, 'Chennai')

standard deviation -  0.32


city_name          Chennai                                              \
sample_category    control                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      52.06      49.23       46.4      47.28   
2024-08-12                      100.0      50.66      46.06      46.62   
2024-08-19                                 100.0      49.37      48.08   
2024-08-26                                            100.0      51.84   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           47.16      44.39      42.69      44.00      43.80   
2024-08-12           46.55      43.68      41.80      43.26      43.04   
2024-08-19           47.68      44.71      42.80      43.69      43.40   
2024-08-26           49.64      46.44      44.46      45.34      44.64   
2024-09-02           50.85      45.87      43.44      44.56      44.05   
2024-09-09           100.0      48.98      44.40      44.89      44.55   
2024-09-16                      100.0      48.81      47.53      46.33   
2024-09-23                                100.00      51.83      48.69   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           39.62      42.76      40.40  
2024-08-12           38.88      41.70      39.95  
2024-08-19           39.29      42.49      40.00  
2024-08-26           40.48      43.70      41.09  
2024-09-02           39.54      42.64      40.59  
2024-09-09           40.00      42.96      40.57  
2024-09-16           41.84      44.85      41.86  
2024-09-23           43.50      46.81      43.28

city_name          Chennai                                              \
sample_category       test                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      52.21      49.23      45.91      47.01   
2024-08-12                      100.0       50.1      45.32      46.02   
2024-08-19                                 100.0      49.15      47.63   
2024-08-26                                            100.0      51.56   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           46.91      44.13      42.93      44.04      43.30   
2024-08-12           46.14      43.74      41.78      42.66      42.44   
2024-08-19           47.13      44.98      42.77      44.00      43.15   
2024-08-26           49.06      46.67      44.59      45.50      44.24   
2024-09-02           50.74      45.83      42.99      44.61      43.65   
2024-09-09           100.0       48.8      44.34      44.85      43.96   
2024-09-16                      100.0      49.06      47.53      46.21   
2024-09-23                                100.00      51.79      48.56   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           38.57      42.38      40.00  
2024-08-12           37.77      41.34      39.18  
2024-08-19           39.10      42.51      39.74  
2024-08-26           40.07      43.90      41.05  
2024-09-02           39.22      42.24      40.08  
2024-09-09           39.45      42.91      40.11  
2024-09-16           41.26      44.60      41.85  
2024-09-23           42.46      46.41      42.57

city_name                       Chennai                                   \
difference     delta - (test - control)                                    
retention_date               2024-08-05 2024-08-12 2024-08-19 2024-08-26   
start_date                                                                 
2024-08-05                          0.0       0.15        0.0      -0.49   
2024-08-12                                     0.0      -0.56      -0.74   
2024-08-19                                                0.0      -0.22   
2024-08-26                                                           0.0   
2024-09-02                                                                 
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

city_name                                                              \
difference                                                              
retention_date 2024-09-02 2024-09-09 2024-09-16 2024-09-23 2024-09-30   
start_date                                                              
2024-08-05          -0.27      -0.25      -0.26       0.24       0.04   
2024-08-12           -0.6      -0.41       0.06      -0.02      -0.60   
2024-08-19          -0.45      -0.55       0.27      -0.03       0.31   
2024-08-26          -0.28      -0.58       0.23       0.13       0.16   
2024-09-02            0.0      -0.11      -0.04      -0.45       0.05   
2024-09-09                       0.0      -0.18      -0.06      -0.04   
2024-09-16                                  0.0       0.25       0.00   
2024-09-23                                            0.00      -0.04   

city_name                                                   
difference                                                  
retention_date 2024-10-07 2024-10-14 2024-10-21 2024-10-28  
start_date                                                  
2024-08-05          -0.50      -1.05      -0.38      -0.40  
2024-08-12          -0.60      -1.11      -0.36      -0.77  
2024-08-19          -0.25      -0.19       0.02      -0.26  
2024-08-26          -0.40      -0.41       0.20      -0.04  
2024-09-02          -0.40      -0.32      -0.40      -0.51  
2024-09-09          -0.59      -0.55      -0.05      -0.46  
2024-09-16          -0.12      -0.58      -0.25      -0.01  
2024-09-23          -0.13      -1.04      -0.40      -0.71

### Delhi

In [144]:
get_city_retention_cohort(df_overall_data, 'Delhi')

standard deviation -  0.34


city_name            Delhi                                              \
sample_category    control                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      51.55      48.41      46.72      45.81   
2024-08-12                      100.0      49.37      45.66      44.23   
2024-08-19                                100.00      49.80      47.12   

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           44.84      43.61      42.29      41.92      42.32   
2024-08-12           43.34      42.09      40.63      40.48      40.93   
2024-08-19           45.84      44.42      42.72      42.31      42.80   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           42.37      41.74      35.55  
2024-08-12           40.79      40.30      35.18  
2024-08-19           42.78      42.19      36.93

city_name            Delhi                                              \
sample_category       test                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      51.65      47.99      46.91      45.94   
2024-08-12                      100.0      49.28      45.82      44.26   
2024-08-19                                100.00      50.69      47.73   

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           45.06      43.87      42.04      41.76      42.11   
2024-08-12           43.35      42.72      40.35      40.39      40.90   
2024-08-19           46.70      45.14      42.99      42.56      43.35   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           42.62      41.82      35.63  
2024-08-12           41.44      40.55      35.34  
2024-08-19           43.65      42.91      37.22

city_name                         Delhi                                   \
difference     delta - (test - control)                                    
retention_date               2024-08-05 2024-08-12 2024-08-19 2024-08-26   
start_date                                                                 
2024-08-05                          0.0        0.1      -0.42       0.19   
2024-08-12                                     0.0      -0.09       0.16   
2024-08-19                                               0.00       0.89   

city_name                                                              \
difference                                                              
retention_date 2024-09-02 2024-09-09 2024-09-16 2024-09-23 2024-09-30   
start_date                                                              
2024-08-05           0.13       0.22       0.26      -0.25      -0.16   
2024-08-12           0.03       0.01       0.63      -0.28      -0.09   
2024-08-19           0.61       0.86       0.72       0.27       0.25   

city_name                                                   
difference                                                  
retention_date 2024-10-07 2024-10-14 2024-10-21 2024-10-28  
start_date                                                  
2024-08-05          -0.21       0.25       0.08       0.08  
2024-08-12          -0.03       0.65       0.25       0.16  
2024-08-19           0.55       0.87       0.72       0.29

### Hyderabad

In [145]:
get_city_retention_cohort(df_overall_data, 'Hyderabad')

standard deviation -  0.24


city_name        Hyderabad                                              \
sample_category    control                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      50.72      48.09      44.07      41.35   
2024-08-12                      100.0      49.53       43.6      40.59   
2024-08-19                                 100.0      45.75      40.91   
2024-08-26                                            100.0      46.19   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           41.91      40.67      40.75      41.09      39.14   
2024-08-12           41.01      39.72      39.60      40.18      38.51   
2024-08-19           40.91      39.65      39.39      39.75      38.05   
2024-08-26           44.29      42.47      42.26      42.24      39.90   
2024-09-02           48.55      44.81      43.95      44.09      41.52   
2024-09-09           100.0      48.07      45.20      44.64      41.91   
2024-09-16                      100.0      48.48      45.95      42.70   
2024-09-23                                100.00      49.21      43.84   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           40.89      39.30      38.55  
2024-08-12           40.07      38.15      37.91  
2024-08-19           39.94      37.81      37.39  
2024-08-26           41.77      39.98      39.26  
2024-09-02           43.16      41.02      40.62  
2024-09-09           43.68      41.55      40.71  
2024-09-16           44.36      42.21      41.10  
2024-09-23           45.49      43.59      41.94

city_name        Hyderabad                                              \
sample_category       test                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      50.61      47.67      43.53      41.25   
2024-08-12                      100.0       49.2      43.56      40.74   
2024-08-19                                 100.0      45.97      40.84   
2024-08-26                                            100.0      46.39   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           41.59      40.17      40.47      40.86      39.42   
2024-08-12           41.23      39.67      39.73      40.36      38.76   
2024-08-19           41.18      39.42      39.24      39.93      38.26   
2024-08-26           44.52      42.28      42.34      42.69      40.09   
2024-09-02           48.49      44.68      44.04      44.46      41.60   
2024-09-09           100.0      47.91      45.35      44.97      42.06   
2024-09-16                      100.0      48.34      46.12      43.12   
2024-09-23                                100.00      49.67      44.06   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           41.15      39.04      38.62  
2024-08-12           40.26      38.04      37.94  
2024-08-19           40.30      37.58      37.47  
2024-08-26           41.98      40.12      39.23  
2024-09-02           43.57      40.93      40.72  
2024-09-09           44.20      41.83      40.66  
2024-09-16           44.93      42.54      41.38  
2024-09-23           46.08      43.53      41.84

city_name                     Hyderabad                                   \
difference     delta - (test - control)                                    
retention_date               2024-08-05 2024-08-12 2024-08-19 2024-08-26   
start_date                                                                 
2024-08-05                          0.0      -0.11      -0.42      -0.54   
2024-08-12                                     0.0      -0.33      -0.04   
2024-08-19                                                0.0       0.22   
2024-08-26                                                           0.0   
2024-09-02                                                                 
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

city_name                                                              \
difference                                                              
retention_date 2024-09-02 2024-09-09 2024-09-16 2024-09-23 2024-09-30   
start_date                                                              
2024-08-05           -0.1      -0.32       -0.5      -0.28      -0.23   
2024-08-12           0.15       0.22      -0.05       0.13       0.18   
2024-08-19          -0.07       0.27      -0.23      -0.15       0.18   
2024-08-26            0.2       0.23      -0.19       0.08       0.45   
2024-09-02            0.0      -0.06      -0.13       0.09       0.37   
2024-09-09                       0.0      -0.16       0.15       0.33   
2024-09-16                                  0.0      -0.14       0.17   
2024-09-23                                            0.00       0.46   

city_name                                                   
difference                                                  
retention_date 2024-10-07 2024-10-14 2024-10-21 2024-10-28  
start_date                                                  
2024-08-05           0.28       0.26      -0.26       0.07  
2024-08-12           0.25       0.19      -0.11       0.03  
2024-08-19           0.21       0.36      -0.23       0.08  
2024-08-26           0.19       0.21       0.14      -0.03  
2024-09-02           0.08       0.41      -0.09       0.10  
2024-09-09           0.15       0.52       0.28      -0.05  
2024-09-16           0.42       0.57       0.33       0.28  
2024-09-23           0.22       0.59      -0.06      -0.10

### Jaipur

In [146]:
get_city_retention_cohort(df_overall_data, 'Jaipur')

standard deviation -  0.55


city_name           Jaipur                                              \
sample_category    control                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      41.42      36.34      36.18      35.61   
2024-08-12                      100.0       39.1      36.49      35.52   
2024-08-19                                 100.0      40.35      37.05   
2024-08-26                                            100.0      41.83   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           35.27      33.04      32.64      32.09      32.17   
2024-08-12           35.33      32.83      32.32      31.75      32.12   
2024-08-19           36.49      34.31      33.32      32.84      33.12   
2024-08-26           38.93      36.28      35.20      34.51      34.20   
2024-09-02           42.68      37.74      36.28      35.47      35.06   
2024-09-09           100.0      41.83      37.85      36.42      36.17   
2024-09-16                      100.0      42.69      39.09      38.14   
2024-09-23                                100.00      41.01      37.72   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           31.64      31.61      24.47  
2024-08-12           31.40      31.80      25.55  
2024-08-19           32.29      32.41      26.39  
2024-08-26           33.51      33.41      25.28  
2024-09-02           34.33      33.96      25.89  
2024-09-09           35.06      34.65      26.38  
2024-09-16           37.14      36.34      27.38  
2024-09-23           36.56      36.85      26.58

city_name           Jaipur                                              \
sample_category       test                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      41.12      36.72       36.3      35.17   
2024-08-12                      100.0      39.45      36.83      34.98   
2024-08-19                                 100.0      41.73      37.37   
2024-08-26                                            100.0      40.94   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           35.34      33.65      32.48      33.12      32.35   
2024-08-12           34.64      32.67      32.54      32.55      31.59   
2024-08-19           36.88      35.22      34.46      34.11      33.91   
2024-08-26           39.26      37.13      35.98      35.04      33.92   
2024-09-02           42.65      38.41      36.50      35.82      35.26   
2024-09-09           100.0      42.31      38.06      38.06      35.84   
2024-09-16                      100.0      43.75      40.57      38.70   
2024-09-23                                100.00      41.43      38.50   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           31.61      32.78      24.55  
2024-08-12           31.33      31.87      25.10  
2024-08-19           32.98      33.22      26.40  
2024-08-26           34.12      33.50      25.65  
2024-09-02           34.38      34.72      25.47  
2024-09-09           36.10      35.58      26.16  
2024-09-16           37.69      37.90      27.80  
2024-09-23           36.71      38.00      26.73

city_name                        Jaipur                                   \
difference     delta - (test - control)                                    
retention_date               2024-08-05 2024-08-12 2024-08-19 2024-08-26   
start_date                                                                 
2024-08-05                          0.0       -0.3       0.38       0.12   
2024-08-12                                     0.0       0.35       0.34   
2024-08-19                                                0.0       1.38   
2024-08-26                                                           0.0   
2024-09-02                                                                 
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

city_name                                                              \
difference                                                              
retention_date 2024-09-02 2024-09-09 2024-09-16 2024-09-23 2024-09-30   
start_date                                                              
2024-08-05          -0.44       0.07       0.61      -0.16       1.03   
2024-08-12          -0.54      -0.69      -0.16       0.22       0.80   
2024-08-19           0.32       0.39       0.91       1.14       1.27   
2024-08-26          -0.89       0.33       0.85       0.78       0.53   
2024-09-02            0.0      -0.03       0.67       0.22       0.35   
2024-09-09                       0.0       0.48       0.21       1.64   
2024-09-16                                  0.0       1.06       1.48   
2024-09-23                                            0.00       0.42   

city_name                                                   
difference                                                  
retention_date 2024-10-07 2024-10-14 2024-10-21 2024-10-28  
start_date                                                  
2024-08-05           0.18      -0.03       1.17       0.08  
2024-08-12          -0.53      -0.07       0.07      -0.45  
2024-08-19           0.79       0.69       0.81       0.01  
2024-08-26          -0.28       0.61       0.09       0.37  
2024-09-02           0.20       0.05       0.76      -0.42  
2024-09-09          -0.33       1.04       0.93      -0.22  
2024-09-16           0.56       0.55       1.56       0.42  
2024-09-23           0.78       0.15       1.15       0.15

### Kolkata

In [147]:
get_city_retention_cohort(df_overall_data, 'Kolkata')

standard deviation -  0.52


city_name          Kolkata                                              \
sample_category    control                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0       42.5       38.4      36.51       38.2   
2024-08-12                      100.0      40.34      37.15      38.24   
2024-08-19                                 100.0      41.13      40.61   
2024-08-26                                            100.0      44.93   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           34.74      35.65      33.70      34.88      31.25   
2024-08-12           34.66      35.54      33.57      34.85      31.73   
2024-08-19            36.5      37.21      35.01      36.00      32.29   
2024-08-26           38.66      39.16      36.76      37.48      33.21   
2024-09-02           40.41      39.15      36.26      37.23      32.68   
2024-09-09           100.0      43.34      39.03      39.13      33.84   
2024-09-16                      100.0      41.04      39.74      34.37   
2024-09-23                                100.00      43.44      35.60   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           30.02      29.58      31.74  
2024-08-12           30.24      29.39      31.84  
2024-08-19           30.98      30.33      32.70  
2024-08-26           32.03      31.41      33.72  
2024-09-02           31.02      30.51      32.85  
2024-09-09           32.48      32.00      34.02  
2024-09-16           32.64      31.88      34.00  
2024-09-23           33.84      33.29      35.13

city_name          Kolkata                                              \
sample_category       test                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      42.26      38.16      35.97      38.61   
2024-08-12                      100.0      38.86      37.04      38.92   
2024-08-19                                 100.0      41.67      41.49   
2024-08-26                                            100.0      44.54   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           33.99       35.7      33.58      35.19      32.00   
2024-08-12            34.5      36.49      33.58      34.33      32.39   
2024-08-19           36.89      37.81      35.94      36.95      33.49   
2024-08-26           38.27      40.01      36.61      37.56      33.85   
2024-09-02           39.44      38.96      35.89      37.22      33.21   
2024-09-09           100.0      43.22      38.14      39.48      33.73   
2024-09-16                      100.0      40.84      39.87      34.81   
2024-09-23                                100.00      43.12      35.95   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           29.68      29.80      32.04  
2024-08-12           30.21      29.69      32.69  
2024-08-19           30.97      31.53      32.69  
2024-08-26           31.87      31.77      33.33  
2024-09-02           30.76      30.54      32.46  
2024-09-09           32.69      33.01      33.92  
2024-09-16           32.04      32.06      33.81  
2024-09-23           33.16      34.12      34.88

city_name                       Kolkata                                   \
difference     delta - (test - control)                                    
retention_date               2024-08-05 2024-08-12 2024-08-19 2024-08-26   
start_date                                                                 
2024-08-05                          0.0      -0.24      -0.24      -0.54   
2024-08-12                                     0.0      -1.48      -0.11   
2024-08-19                                                0.0       0.54   
2024-08-26                                                           0.0   
2024-09-02                                                                 
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

city_name                                                              \
difference                                                              
retention_date 2024-09-02 2024-09-09 2024-09-16 2024-09-23 2024-09-30   
start_date                                                              
2024-08-05           0.41      -0.75       0.05      -0.12       0.31   
2024-08-12           0.68      -0.16       0.95       0.01      -0.52   
2024-08-19           0.88       0.39        0.6       0.93       0.95   
2024-08-26          -0.39      -0.39       0.85      -0.15       0.08   
2024-09-02            0.0      -0.97      -0.19      -0.37      -0.01   
2024-09-09                       0.0      -0.12      -0.89       0.35   
2024-09-16                                  0.0      -0.20       0.13   
2024-09-23                                            0.00      -0.32   

city_name                                                   
difference                                                  
retention_date 2024-10-07 2024-10-14 2024-10-21 2024-10-28  
start_date                                                  
2024-08-05           0.75      -0.34       0.22       0.30  
2024-08-12           0.66      -0.03       0.30       0.85  
2024-08-19           1.20      -0.01       1.20      -0.01  
2024-08-26           0.64      -0.16       0.36      -0.39  
2024-09-02           0.53      -0.26       0.03      -0.39  
2024-09-09          -0.11       0.21       1.01      -0.10  
2024-09-16           0.44      -0.60       0.18      -0.19  
2024-09-23           0.35      -0.68       0.83      -0.25

### Mumbai

In [148]:
get_city_retention_cohort(df_overall_data, 'Mumbai')

standard deviation -  0.74


city_name           Mumbai                                              \
sample_category    control                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      45.32      43.24      40.76      40.82   
2024-08-12                      100.0      44.76      40.32      40.14   
2024-08-19                                 100.0       41.9      40.22   
2024-08-26                                            100.0      45.62   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           39.18      36.64      38.15      36.23      36.98   
2024-08-12           38.51      36.02      37.01      35.30      36.17   
2024-08-19           38.32      35.48      36.57      34.60      35.44   
2024-08-26           41.71      38.69      39.89      37.62      38.04   
2024-09-02            43.6      38.42      39.18      37.01      37.68   
2024-09-09           100.0      41.27      40.08      37.44      38.15   
2024-09-16                      100.0      46.00      41.74      41.46   
2024-09-23                                100.00      43.04      41.62   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           36.21      35.91      32.58  
2024-08-12           35.34      34.79      32.30  
2024-08-19           34.66      34.31      31.75  
2024-08-26           37.24      36.82      33.25  
2024-09-02           36.54      35.92      33.04  
2024-09-09           36.86      36.11      33.13  
2024-09-16           40.47      39.37      35.37  
2024-09-23           40.25      38.99      34.63

city_name           Mumbai                                              \
sample_category       test                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      45.99      44.09      41.12      41.24   
2024-08-12                      100.0      45.96       41.4       40.3   
2024-08-19                                 100.0       42.8      40.95   
2024-08-26                                            100.0       45.7   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           40.55      38.19      39.51      37.58      37.96   
2024-08-12           39.53       36.9      38.82      37.39      37.02   
2024-08-19           39.67      36.86      38.01      36.47      36.56   
2024-08-26           42.94      39.65      40.63      39.10      38.39   
2024-09-02           44.49      38.88      39.24      39.08      37.80   
2024-09-09           100.0      41.65      41.37      39.68      38.41   
2024-09-16                      100.0      47.58      43.90      40.80   
2024-09-23                                100.00      45.01      41.92   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           37.61      35.08      32.55  
2024-08-12           36.50      34.90      32.44  
2024-08-19           36.44      34.81      33.38  
2024-08-26           38.12      36.99      33.80  
2024-09-02           37.64      35.78      33.29  
2024-09-09           38.10      35.98      33.70  
2024-09-16           41.19      38.35      35.25  
2024-09-23           40.58      38.57      34.68

city_name                        Mumbai                                   \
difference     delta - (test - control)                                    
retention_date               2024-08-05 2024-08-12 2024-08-19 2024-08-26   
start_date                                                                 
2024-08-05                          0.0       0.67       0.85       0.36   
2024-08-12                                     0.0        1.2       1.08   
2024-08-19                                                0.0        0.9   
2024-08-26                                                           0.0   
2024-09-02                                                                 
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

city_name                                                              \
difference                                                              
retention_date 2024-09-02 2024-09-09 2024-09-16 2024-09-23 2024-09-30   
start_date                                                              
2024-08-05           0.42       1.37       1.55       1.36       1.35   
2024-08-12           0.16       1.02       0.88       1.81       2.09   
2024-08-19           0.73       1.35       1.38       1.44       1.87   
2024-08-26           0.08       1.23       0.96       0.74       1.48   
2024-09-02            0.0       0.89       0.46       0.06       2.07   
2024-09-09                       0.0       0.38       1.29       2.24   
2024-09-16                                  0.0       1.58       2.16   
2024-09-23                                            0.00       1.97   

city_name                                                   
difference                                                  
retention_date 2024-10-07 2024-10-14 2024-10-21 2024-10-28  
start_date                                                  
2024-08-05           0.98       1.40      -0.83      -0.03  
2024-08-12           0.85       1.16       0.11       0.14  
2024-08-19           1.12       1.78       0.50       1.63  
2024-08-26           0.35       0.88       0.17       0.55  
2024-09-02           0.12       1.10      -0.14       0.25  
2024-09-09           0.26       1.24      -0.13       0.57  
2024-09-16          -0.66       0.72      -1.02      -0.12  
2024-09-23           0.30       0.33      -0.42       0.05

<div class="alert alert-block alert-info">
    
## Segmentation level Analusis
</div>

In [167]:
execution_name_list = ['holdout_segment_ban', 'holdout_segment_che', 'holdout_segment_hyd', 'holdout_segment_jai', 
                       'holdout_segment_kol', 'holdout_segment_mum']

In [161]:
print(experiment_name)
print(week_start_date)
print(group_tc)
print(retention_segment)
print(execution_city_name_mapping)

Ads-Holdout-Group-Experiment
['2024-08-05', '2024-08-12', '2024-08-19', '2024-08-26', '2024-09-02', '2024-09-09', '2024-09-16', '2024-09-23']
['control', 'test']
['HH', 'GOLD', 'PLATINUM', 'DORMANT', 'UNKNOWN', 'ELITE', 'SILVER', 'PRIME']
{'holdout_segment_ban': 'Bangalore', 'holdout_segment_che': 'Chennai', 'holdout_segment_del': 'Delhi', 'holdout_segment_hyd': 'Hyderabad', 'holdout_segment_jai': 'Jaipur', 'holdout_segment_kol': 'Kolkata', 'holdout_segment_mum': 'Mumbai'}


In [208]:
execution_name_list = ['holdout_segment_hyd']

In [209]:
def get_weekly_segment_retention_data(experiment_name, execution_name_list, group_tc, week_start_date, retention_segment):

    df = pd.DataFrame()

    for execution_name in execution_name_list:
    
        for date in week_start_date:
    
            for sample_category in group_tc:
    
                for segment in retention_segment:
                
                    base_query = f"""
            
                    WITH base AS (
                            SELECT 
                                DISTINCT
                                DATE_FORMAT(DATE_TRUNC('WEEK',DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y-%m-%d') yyyymmdd_base_week,
                                sample_category,
                                customer_id,
                                execution_name 
                            FROM
                                iceberg.experiments_internal.rapidex_user_customer_adslots_v1_root
                            WHERE
                                yyyymmdd >= DATE_FORMAT(DATE_TRUNC('WEEK', DATE('{date}')), '%Y%m%d')
                                AND yyyymmdd <= DATE_FORMAT(DATE_TRUNC('WEEK', DATE('{date}')) + INTERVAL '6' DAY , '%Y%m%d')
                                AND experiment_name = '{experiment_name}'
                                AND sample_category = '{sample_category}'
                                AND execution_name = '{execution_name}'
                        ),
    
                        segment AS (
                            SELECT
                                customer_id user_id,
                                taxi_retention_segments retention_segment 
                            FROM 
                                datasets.iallocator_customer_segments
                            WHERE 
                                run_date = DATE_FORMAT(DATE_TRUNC('WEEK', DATE('{date}')) + INTERVAL '6' DAY , '%Y-%m-%d')
                                AND taxi_retention_segments = '{segment}'
                        ),
    
                        base_customer AS (
                            SELECT 
                                base.*,
                                retention_segment
                            FROM
                                base
                            JOIN 
                                segment
                                ON customer_id = user_id
                        ),
                        
                        activity AS (
                            SELECT 
                                DISTINCT
                                DATE_FORMAT(DATE_TRUNC('week', DATE_PARSE(day, '%Y-%m-%d')),'%Y-%m-%d') as date_period,
                                yyyymmdd_base_week,
                                customerid customer_id,
                                execution_name,
                                sample_category,
                                retention_segment
                            FROM 
                                datasets.customer_rf_daily_kpi
                            JOIN 
                                base_customer
                                ON customer_id = customerid
                            WHERE 
                                day >= DATE_FORMAT(DATE_TRUNC('WEEK', DATE('{date}')), '%Y-%m-%d')
                                AND day <= DATE_FORMAT(DATE_TRUNC('WEEK', CURRENT_DATE) - INTERVAL '1' DAY, '%Y-%m-%d')
                                AND service_name IN ('Link', 'CabPremium', 'Auto Pool', 'Bike Lite', 'Auto', 'Bike Metro', 'CabEconomy', 'Auto NCR', 'Bike Pink')
                                AND net_rides_daily > 0
                            -- GROUP BY 1,2,3
                        ),
                        
                        customer_totals AS (
                            SELECT 
                                execution_name,
                                sample_category,
                                retention_segment,
                                date_period, 
                                COUNT(DISTINCT customer_id) AS total_cx
                            FROM 
                                activity
                            WHERE 
                                date_period = yyyymmdd_base_week
                            GROUP BY 1,2,3,4
                        ),
                        
                        customer_counts AS (
                            SELECT 
                                a1.execution_name, 
                                a1.sample_category,
                                a1.retention_segment,
                                a1.date_period AS date_period1,
                                a2.date_period AS date_period2,
                                COUNT(DISTINCT a1.customer_id) AS count_t1,
                                COUNT(DISTINCT a2.customer_id) AS count_t2
                            FROM 
                                activity a1
                            LEFT JOIN 
                                activity a2 
                                ON a1.customer_id = a2.customer_id 
                                AND a1.execution_name = a2.execution_name
                                AND a1.sample_category = a2.sample_category
                                AND a1.retention_segment = a2.retention_segment
                                AND a1.date_period <= a2.date_period
                            GROUP BY 1,2,3,4,5
                        )
                        
                        SELECT 
                            customer_counts.execution_name,
                            customer_counts.sample_category,
                            customer_counts.retention_segment,
                            date_period1 AS start_date, 
                            date_period2 AS retention_date, 
                            CASE 
                                WHEN date_period1 = date_period2 THEN 100.00 
                                ELSE (100.00 * customer_counts.count_t1) / customer_totals.total_cx 
                            END AS retention_percentage
                        FROM 
                            customer_counts 
                        JOIN 
                            customer_totals 
                            ON customer_totals.date_period = customer_counts.date_period1
                            AND customer_totals.execution_name = customer_counts.execution_name
                            AND customer_totals.sample_category = customer_counts.sample_category
                            AND customer_totals.retention_segment = customer_counts.retention_segment
                        WHERE 
                            date_period2 IS NOT NULL
                        ORDER BY 1,2,3,5,4
            
                    """
                    
                    df_base = pd.read_sql(base_query, connection)
                    df = pd.concat([df, df_base], ignore_index=True)
                    df.to_csv(local_extract + '/city_segment_level_experiment_retention_data.csv', index=False)
                    print(segment, ' - ', sample_category, ' data fetched successfully for ', date)

    return df

In [210]:
df_segment_data = get_weekly_segment_retention_data(experiment_name,execution_name_list, group_tc, week_start_date, retention_segment)

HH  -  control  data fetched successfully for  2024-08-05
GOLD  -  control  data fetched successfully for  2024-08-05
PLATINUM  -  control  data fetched successfully for  2024-08-05
DORMANT  -  control  data fetched successfully for  2024-08-05
UNKNOWN  -  control  data fetched successfully for  2024-08-05
ELITE  -  control  data fetched successfully for  2024-08-05
SILVER  -  control  data fetched successfully for  2024-08-05
PRIME  -  control  data fetched successfully for  2024-08-05
HH  -  test  data fetched successfully for  2024-08-05
GOLD  -  test  data fetched successfully for  2024-08-05


DatabaseError: {'message': 'Error fetching http://10.32.124.2:8080/v1/task/20241108_043038_00268_s6b6z.2.13.0/results/1/0: Expected response code to be 200, but was 503:\n<html>\n<head>\n<meta http-equiv="Content-Type" content="text/html;charset=ISO-8859-1"/>\n<title>Error 503 Service Unavailable</title>\n</head>\n<body><h2>HTTP ERROR 503 Service Unavailable</h2>\n<table>\n<tr><th>URI:</th><td>/v1/task/20241108_043038_00268_s6b6z.2.13.0/results/1/0</td></tr>\n<tr><th>STATUS:</th><td>503</td></tr>\n<tr><th>MESSAGE:</th><td>Service Unavailable</td></tr>\n<tr><th>SERVLET:</th><td>-</td></tr>\n</table>\n\n</body>\n</html>\n', 'errorCode': 65539, 'errorName': 'PAGE_TRANSPORT_ERROR', 'errorType': 'INTERNAL_ERROR', 'failureInfo': {'type': 'io.trino.operator.PageTransportErrorException', 'message': 'Error fetching http://10.32.124.2:8080/v1/task/20241108_043038_00268_s6b6z.2.13.0/results/1/0: Expected response code to be 200, but was 503:\n<html>\n<head>\n<meta http-equiv="Content-Type" conten

In [ ]:
df_segment_data.to_csv(local_extract + '/city_segment_level_experiment_retention_data.csv', index=False)

In [175]:
df_segment_data = pd.read_csv(local_extract + '/city_segment_level_experiment_retention_data_v1.csv')
print(df_segment_data.shape)
df_segment_data['city_name'] = df_segment_data['execution_name'].map(execution_city_name_mapping)
df_segment_data.head(5)

(1731, 6)


,execution_name,sample_category,retention_segment,start_date,retention_date,retention_percentage,city_name
0,holdout_segment_ban,control,HH,2024-08-05,2024-08-05,100.00,Bangalore
1,holdout_segment_ban,control,HH,2024-08-05,2024-08-12,26.21,Bangalore
2,holdout_segment_ban,control,HH,2024-08-05,2024-08-19,21.92,Bangalore
3,holdout_segment_ban,control,HH,2024-08-05,2024-08-26,19.69,Bangalore
4,holdout_segment_ban,control,HH,2024-08-05,2024-09-02,19.55,Bangalore


In [196]:
def get_city_segment_retention_cohort(df, city_name, segment):
    
    df_control = df[(df['city_name'] == city_name) & (df['sample_category'] == 'control') & (df['retention_segment'] == segment) ]
    df_test = df[(df['city_name'] == city_name) & (df['sample_category'] == 'test') & (df['retention_segment'] == segment)]
    df_merge = pd.merge(df_control,df_test, how='inner',on=['city_name', 'start_date', 'retention_date'])
    df_merge['retention_delta'] = df_merge['retention_percentage_y'] - df_merge['retention_percentage_x']
    df_merge['difference'] = 'delta - (test - control)'
    df1 = df_control.pivot(index=['start_date'], 
                           columns=['retention_segment', 'sample_category', 'retention_date'],
                           values='retention_percentage').fillna('')
    df2 = df_test.pivot(index=['start_date'], 
                        columns=['retention_segment', 'sample_category', 'retention_date'],
                        values='retention_percentage').fillna('')
    df3 = df_merge.pivot(index=['start_date'], 
                         columns=['city_name', 'difference', 'retention_date'], 
                         values='retention_delta').fillna('')
    print('standard deviation - ', round(df_merge['retention_delta'].std(), 2))
    
    return display(df1,df2,df3)

In [197]:
def city_segment_analysis(df, city, retention_segment):
    
    for segment in retention_segment:
        
        get_city_segment_retention_cohort(df, city, segment)

### Bangalore

In [198]:
city_segment_analysis(df_segment_data, 'Bangalore', retention_segment)

standard deviation -  0.57


retention_segment         HH                                              \
sample_category      control                                               
retention_date    2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                                 
2024-08-05             100.0      26.21      21.92      19.69      19.55   
2024-08-12                        100.0      24.11      19.57      19.06   
2024-08-19                                   100.0       23.9      20.46   
2024-08-26                                              100.0      24.32   
2024-09-02                                                         100.0   
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

retention_segment                                                         \
sample_category                                                            
retention_date    2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                                 
2024-08-05             19.21      17.99      17.03      17.86      18.67   
2024-08-12             18.67      17.46      16.09      17.04      18.02   
2024-08-19             19.62      18.18      17.06      17.54      17.99   
2024-08-26             20.89      19.18      17.84      18.40      18.33   
2024-09-02             23.89      19.81      17.63      18.56      19.20   
2024-09-09             100.0      23.79      18.93      18.84      19.03   
2024-09-16                        100.0      22.78      20.06      19.71   
2024-09-23                                  100.00      24.15      21.02   

retention_segment                                   
sample_category                                     
retention_date    2024-10-14 2024-10-21 2024-10-28  
start_date                                          
2024-08-05             17.81      17.04      16.29  
2024-08-12             17.23      16.30      16.17  
2024-08-19             17.47      16.55      16.12  
2024-08-26             17.75      17.01      16.32  
2024-09-02             17.55      16.51      16.87  
2024-09-09             17.99      16.81      16.39  
2024-09-16             18.31      17.04      16.48  
2024-09-23             19.23      18.08      16.88

retention_segment         HH                                              \
sample_category         test                                               
retention_date    2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                                 
2024-08-05             100.0       27.2      21.77      19.61      19.19   
2024-08-12                        100.0      23.92      20.18       20.3   
2024-08-19                                   100.0      24.51      19.91   
2024-08-26                                              100.0      24.34   
2024-09-02                                                         100.0   
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

retention_segment                                                         \
sample_category                                                            
retention_date    2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                                 
2024-08-05             19.02       18.0      17.36      17.08      17.17   
2024-08-12              19.3      18.05      16.44      17.50      18.46   
2024-08-19             19.78      18.48      18.14      16.63      17.81   
2024-08-26             21.62       18.8      17.72      18.37      18.43   
2024-09-02             23.71      19.63      17.11      17.67      19.63   
2024-09-09             100.0      23.21      18.10      18.99      20.66   
2024-09-16                        100.0      22.81      20.38      19.65   
2024-09-23                                  100.00      24.06      19.84   

retention_segment                                   
sample_category                                     
retention_date    2024-10-14 2024-10-21 2024-10-28  
start_date                                          
2024-08-05             18.39      16.64      16.50  
2024-08-12             17.71      16.11      16.28  
2024-08-19             17.71      16.17      16.71  
2024-08-26             17.55      18.00      17.13  
2024-09-02             17.40      15.82      16.51  
2024-09-09             18.65      16.61      16.25  
2024-09-16             18.73      18.73      16.73  
2024-09-23             19.19      18.61      16.76

city_name                     Bangalore                                   \
difference     delta - (test - control)                                    
retention_date               2024-08-05 2024-08-12 2024-08-19 2024-08-26   
start_date                                                                 
2024-08-05                          0.0       0.99      -0.15      -0.08   
2024-08-12                                     0.0      -0.19       0.61   
2024-08-19                                                0.0       0.61   
2024-08-26                                                           0.0   
2024-09-02                                                                 
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

city_name                                                              \
difference                                                              
retention_date 2024-09-02 2024-09-09 2024-09-16 2024-09-23 2024-09-30   
start_date                                                              
2024-08-05          -0.36      -0.19       0.01       0.33      -0.78   
2024-08-12           1.24       0.63       0.59       0.35       0.46   
2024-08-19          -0.55       0.16        0.3       1.08      -0.91   
2024-08-26           0.02       0.73      -0.38      -0.12      -0.03   
2024-09-02            0.0      -0.18      -0.18      -0.52      -0.89   
2024-09-09                       0.0      -0.58      -0.83       0.15   
2024-09-16                                  0.0       0.03       0.32   
2024-09-23                                            0.00      -0.09   

city_name                                                   
difference                                                  
retention_date 2024-10-07 2024-10-14 2024-10-21 2024-10-28  
start_date                                                  
2024-08-05          -1.50       0.58      -0.40       0.21  
2024-08-12           0.44       0.48      -0.19       0.11  
2024-08-19          -0.18       0.24      -0.38       0.59  
2024-08-26           0.10      -0.20       0.99       0.81  
2024-09-02           0.43      -0.15      -0.69      -0.36  
2024-09-09           1.63       0.66      -0.20      -0.14  
2024-09-16          -0.06       0.42       1.69       0.25  
2024-09-23          -1.18      -0.04       0.53      -0.12

standard deviation -  0.52


...

city_name                     Bangalore                                   \
difference     delta - (test - control)                                    
retention_date               2024-08-05 2024-08-12 2024-08-19 2024-08-26   
start_date                                                                 
2024-08-05                          0.0      -0.15      -0.11       0.17   
2024-08-12                                     0.0        0.0      -0.55   
2024-08-19                                                0.0       0.04   
2024-08-26                                                           0.0   
2024-09-02                                                                 
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

city_name                                                              \
difference                                                              
retention_date 2024-09-02 2024-09-09 2024-09-16 2024-09-23 2024-09-30   
start_date                                                              
2024-08-05          -0.32       0.04       0.73      -0.18      -0.06   
2024-08-12           0.73      -0.06       0.99       0.25       0.99   
2024-08-19          -0.04       -0.8       1.07       0.11       0.33   
2024-08-26          -1.93       0.62      -0.08      -0.16      -1.26   
2024-09-02            0.0       0.22       0.35      -0.96       0.31   
2024-09-09                       0.0       -0.1      -0.63      -0.65   
2024-09-16                                  0.0       0.25       0.64   
2024-09-23                                            0.00      -0.96   

city_name                                                   
difference                                                  
retention_date 2024-10-07 2024-10-14 2024-10-21 2024-10-28  
start_date                                                  
2024-08-05          -0.95       0.76       0.16      -0.92  
2024-08-12          -0.21      -0.40       0.07       0.50  
2024-08-19           1.51       0.81       0.68       0.80  
2024-08-26           0.31      -1.42      -0.72       0.90  
2024-09-02           0.56      -0.28      -0.05       0.10  
2024-09-09           0.01       0.20       0.00       0.12  
2024-09-16           0.62       0.55      -0.40       1.15  
2024-09-23          -0.69      -0.33      -0.90      -0.48

standard deviation -  3.42


retention_segment      PRIME                                              \
sample_category      control                                               
retention_date    2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                                 
2024-08-05             100.0       80.5      77.33      76.64      72.91   
2024-08-12                        100.0      81.14      78.25      74.01   
2024-08-19                                   100.0      82.22      77.24   
2024-08-26                                              100.0      81.06   
2024-09-02                                                         100.0   
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

retention_segment                                                         \
sample_category                                                            
retention_date    2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                                 
2024-08-05             71.91      70.67      69.32      67.62      65.08   
2024-08-12             73.64      70.96      69.69      68.76      65.76   
2024-08-19             75.27      73.42      71.70      69.87      66.78   
2024-08-26             77.93       75.6      73.34      70.96      68.34   
2024-09-02             82.34      78.86      76.47      73.13      69.78   
2024-09-09             100.0      82.72      79.07      75.21      70.95   
2024-09-16                        100.0      83.51      78.01      73.15   
2024-09-23                                  100.00      81.97      75.88   

retention_segment                                   
sample_category                                     
retention_date    2024-10-14 2024-10-21 2024-10-28  
start_date                                          
2024-08-05             63.62      62.78      55.99  
2024-08-12             64.49      63.45      57.69  
2024-08-19             64.67      63.96      57.45  
2024-08-26             65.85      64.58      57.46  
2024-09-02             67.23      65.93      58.81  
2024-09-09             68.79      66.76      58.84  
2024-09-16             70.35      68.16      60.05  
2024-09-23             72.85      69.80      61.27

retention_segment      PRIME                                              \
sample_category         test                                               
retention_date    2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                                 
2024-08-05             100.0      84.21      76.32      72.11      71.58   
2024-08-12                        100.0      81.62      79.46      78.92   
2024-08-19                                   100.0      82.08      79.77   
2024-08-26                                              100.0      81.11   
2024-09-02                                                         100.0   
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

retention_segment                                                         \
sample_category                                                            
retention_date    2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                                 
2024-08-05             70.53      71.05      68.42      63.68      58.95   
2024-08-12             75.14      75.68      68.11      63.24      60.00   
2024-08-19             75.72      77.46      67.63      67.05      60.69   
2024-08-26             78.89      78.33      69.44      66.11      60.00   
2024-09-02             87.28      83.82      76.30      71.10      65.32   
2024-09-09             100.0      83.24      75.68      74.05      65.95   
2024-09-16                        100.0      79.78      73.22      65.03   
2024-09-23                                  100.00      80.12      72.89   

retention_segment                                   
sample_category                                     
retention_date    2024-10-14 2024-10-21 2024-10-28  
start_date                                          
2024-08-05             57.89      61.05      58.42  
2024-08-12             60.54      64.32      60.54  
2024-08-19             57.80      61.85      58.96  
2024-08-26             59.44      63.89      55.00  
2024-09-02             65.90      71.68      61.85  
2024-09-09             63.24      69.73      62.16  
2024-09-16             68.31      70.49      59.02  
2024-09-23             73.49      75.30      62.65

city_name                     Bangalore                                   \
difference     delta - (test - control)                                    
retention_date               2024-08-05 2024-08-12 2024-08-19 2024-08-26   
start_date                                                                 
2024-08-05                          0.0       3.71      -1.01      -4.53   
2024-08-12                                     0.0       0.48       1.21   
2024-08-19                                                0.0      -0.14   
2024-08-26                                                           0.0   
2024-09-02                                                                 
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

city_name                                                              \
difference                                                              
retention_date 2024-09-02 2024-09-09 2024-09-16 2024-09-23 2024-09-30   
start_date                                                              
2024-08-05          -1.33      -1.38       0.38      -0.90      -3.94   
2024-08-12           4.91        1.5       4.72      -1.58      -5.52   
2024-08-19           2.53       0.45       4.04      -4.07      -2.82   
2024-08-26           0.05       0.96       2.73      -3.90      -4.85   
2024-09-02            0.0       4.94       4.96      -0.17      -2.03   
2024-09-09                       0.0       0.52      -3.39      -1.16   
2024-09-16                                  0.0      -3.73      -4.79   
2024-09-23                                            0.00      -1.85   

city_name                                                   
difference                                                  
retention_date 2024-10-07 2024-10-14 2024-10-21 2024-10-28  
start_date                                                  
2024-08-05          -6.13      -5.73      -1.73       2.43  
2024-08-12          -5.76      -3.95       0.87       2.85  
2024-08-19          -6.09      -6.87      -2.11       1.51  
2024-08-26          -8.34      -6.41      -0.69      -2.46  
2024-09-02          -4.46      -1.33       5.75       3.04  
2024-09-09          -5.00      -5.55       2.97       3.32  
2024-09-16          -8.12      -2.04       2.33      -1.03  
2024-09-23          -2.99       0.64       5.50       1.38